# 06 Feature Importance And Biological Interpretation

In [2]:
# Notebook 06: Feature importance and biological interpretation

# Initial attempt to back-project PCA importances to gene space.
# This version did not account for the variance filtering step applied before PCA.
# A corrected version is implemented in the cell below.

import joblib
import numpy as np
import pandas as pd

from utils.data_utils import project_root

ROOT = project_root()
IN_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
TAB_DIR = ROOT / "reports" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

# Load data to recover PCA component axes and original feature list
X_aligned = pd.read_parquet(IN_DIR / "X_aligned.parquet")
pca = joblib.load(MODELS_DIR / "pca.joblib")

importance_rows = []
for model_name, model_path in {
    "random_forest": MODELS_DIR / "rf_best.joblib",
    "xgboost": MODELS_DIR / "xgb_best.joblib",
}.items():
    if not model_path.exists():
        continue
    model = joblib.load(model_path)
    if not hasattr(model, "feature_importances_"):
        continue

    pc_importance = model.feature_importances_
    # Back-project importances from PCA-space to gene-space using absolute component loadings.
    gene_importance = np.abs(pca.components_).T @ pc_importance

    top_idx = np.argsort(gene_importance)[::-1][:50]
    top_genes = X_aligned.columns[top_idx]
    top_vals = gene_importance[top_idx]

    df = pd.DataFrame(
        {
            "model": model_name,
            "gene": top_genes,
            "importance": top_vals,
            "rank": np.arange(1, len(top_genes) + 1),
        }
    )
    importance_rows.append(df)

if importance_rows:
    out = pd.concat(importance_rows, ignore_index=True)
    out.to_csv(TAB_DIR / "top_predictive_genes.csv", index=False)
    out.head(20)
else:
    print("No feature importances available yet. Run model notebooks first.")


In [3]:
# Debug sanity Check: Verifying Gene Mapping
# This cell was used to check whether selected EGFR-related genes were present in the processed gene list and 
# whether the PCA back-projection was mapping features correctly. 

import numpy as np
from utils.data_utils import project_root
import pandas as pd

ROOT = project_root()
IN_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

X_aligned = pd.read_parquet(IN_DIR / "X_aligned.parquet")
pca = joblib.load(MODELS_DIR / "pca.joblib")
rf = joblib.load(MODELS_DIR / "rf_best.joblib")

print("X_aligned columns:", len(X_aligned.columns))
print("PCA components shape:", pca.components_.shape)
print("RF feature importances length:", len(rf.feature_importances_))
print("Has feature_importances_:", hasattr(rf, "feature_importances_"))
print("RF model type:", type(rf))

X_aligned columns: 18900
PCA components shape: (100, 18900)
RF feature importances length: 100
Has feature_importances_: True
RF model type: <class 'sklearn.ensemble._forest.RandomForestRegressor'>


In [4]:
# Corrected version: uses variance filter to recover the exact gene names 
# that PCA was trained on, ensuring dimensions match correctly.
# Feature importances from tree models are in PCA space (100 components).
# We back-project them to gene space using the absolute PCA loading matrix,
# giving each gene an approximate importance score.

import joblib
import numpy as np
import pandas as pd
from utils.data_utils import project_root

ROOT = project_root()
IN_DIR = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
TAB_DIR = ROOT / "reports" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

X_aligned = pd.read_parquet(IN_DIR / "X_aligned.parquet")
pca = joblib.load(MODELS_DIR / "pca.joblib")
var_filter = joblib.load(MODELS_DIR / "variance_filter.joblib")

# Recover the gene names that survived variance filtering
genes_after_filter = X_aligned.columns[var_filter.get_support()]
print(f"Genes after variance filter: {len(genes_after_filter)}")
print(f"PCA components shape: {pca.components_.shape}")

importance_rows = []
for model_name, model_path in {
    "random_forest": MODELS_DIR / "rf_best.joblib",
    "xgboost": MODELS_DIR / "xgb_best.joblib",
}.items():
    if not model_path.exists():
        continue
    model = joblib.load(model_path)
    if not hasattr(model, "feature_importances_"):
        continue

    pc_importance = model.feature_importances_
    gene_importance = np.abs(pca.components_).T @ pc_importance

    top_idx = np.argsort(gene_importance)[::-1][:50]
    top_genes = genes_after_filter[top_idx]
    top_vals = gene_importance[top_idx]

    df = pd.DataFrame({
        "model": model_name,
        "gene": top_genes,
        "importance": top_vals,
        "rank": np.arange(1, len(top_genes) + 1),
    })
    importance_rows.append(df)

if importance_rows:
    out = pd.concat(importance_rows, ignore_index=True)
    out.to_csv(TAB_DIR / "top_predictive_genes.csv", index=False)
    display(out.head(20))
else:
    print("No feature importances available.")

Genes after variance filter: 18900
PCA components shape: (100, 18900)


,model,gene,importance,rank
0,random_forest,KIR3DL1,0.007652,1
1,random_forest,VPS41,0.007643,2
2,random_forest,USP42,0.007580,3
3,random_forest,C17orf61,0.007529,4
4,random_forest,C7orf33,0.007492,5
5,random_forest,KIR3DL2,0.007472,6
6,random_forest,SNRNP40,0.007462,7
7,random_forest,PPP4C,0.007461,8
8,random_forest,CD164,0.007437,9
9,random_forest,OTX2,0.007434,10


In [5]:
# Check where known Erlotinib/EGFR pathway genes rank across all 18,900 genes.
# Since these genes were not in the top 50, we look at their full rank and 
# percentile to understand how the model weighted them relative to all genes.
# Results are saved and displayed for interpretation in the markdown below.

egfr_genes = ["EGFR", "ERBB2", "ERBB3", "ERBB4", "MET", 
              "KRAS", "PIK3CA", "AKT1", "MAPK1", "SRC"]

rows_egfr = []
for model_name, model_path in {
    "random_forest": MODELS_DIR / "rf_best.joblib",
    "xgboost": MODELS_DIR / "xgb_best.joblib",
}.items():
    model = joblib.load(model_path)
    pc_importance = model.feature_importances_
    gene_importance = np.abs(pca.components_).T @ pc_importance
    all_importance = pd.Series(gene_importance, index=genes_after_filter)
    all_ranks = all_importance.rank(ascending=False)
    
    for g in egfr_genes:
        if g in all_ranks.index:
            rows_egfr.append({
                "model": model_name,
                "gene": g,
                "rank": int(all_ranks[g]),
                "total_genes": len(all_ranks),
                "importance": all_importance[g],
                "percentile": round(100 * (1 - all_ranks[g]/len(all_ranks)), 1)
            })

egfr_df = pd.DataFrame(rows_egfr)
egfr_df.to_csv(TAB_DIR / "egfr_pathway_gene_ranks.csv", index=False)
display(egfr_df.sort_values(["model", "rank"]))

,model,gene,rank,total_genes,importance,percentile
8,random_forest,MAPK1,1732,18900,0.006468,90.8
7,random_forest,AKT1,3559,18900,0.006245,81.2
9,random_forest,SRC,6990,18900,0.005962,63.0
5,random_forest,KRAS,7932,18900,0.005893,58.0
6,random_forest,PIK3CA,8194,18900,0.005875,56.6
3,random_forest,ERBB4,9301,18900,0.005788,50.8
0,random_forest,EGFR,13547,18900,0.005440,28.3
4,random_forest,MET,14448,18900,0.005359,23.6
2,random_forest,ERBB3,16864,18900,0.005062,10.8
1,random_forest,ERBB2,17290,18900,0.004990,8.5


## Feature Importance and Biological Interpretation — Results

### Why Feature Importance Analysis?
After identifying the best performing model, the next goal is to ask: which genes 
drove the predictions? This analysis helps connect the machine learning results 
back to the biology of Erlotinib — and also helps us understand whether the 
model is learning something meaningful or just picking up noise.

### Method and Design Choices
Feature importances were extracted from Random Forest and XGBoost — the two 
tree-based models — because they natively produce a `feature_importances_` 
attribute that scores how much each feature contributed to reducing prediction 
error across all trees.

However, the models were trained on 100 PCA components, not on the original 
18,900 genes directly. PCA components are combinations of many genes. This 
means a component ranked as important by the model does not directly tell us 
which genes matter. To recover gene-level importance, the PCA component 
importances were **back-projected to gene space** using the absolute values 
of the PCA loading matrix. This gives each gene an approximate importance 
score based on how strongly it contributed to the most predictive PCA components.

This is an approximation — it assumes that genes contributing more to important 
components are more relevant to the prediction. It provides a reasonable and 
interpretable first-pass ranking given the PCA-based pipeline.

### Top 10 Predictive Genes (Random Forest)
| Rank | Gene | Importance |
|---|---|---|
| 1 | KIR3DL1 | 0.00765 |
| 2 | VPS41 | 0.00764 |
| 3 | USP42 | 0.00758 |
| 4 | C17orf61 | 0.00753 |
| 5 | C7orf33 | 0.00749 |
| 6 | KIR3DL2 | 0.00747 |
| 7 | SNRNP40 | 0.00746 |
| 8 | PPP4C | 0.00746 |
| 9 | CD164 | 0.00744 |
| 10 | OTX2 | 0.00743 |

Two things stand out in this table. First, the importance scores are very 
uniform — all around 0.007, with no gene standing out significantly above 
the rest. This tells us that no single gene dominates the prediction. The 
predictive signal is spread across many genes rather than concentrated in 
a few strong predictors. This is common in high-dimensional gene expression data, where many features contribute small effects.

Second, the top genes (KIR3DL1, VPS41, USP42) are not well-known Erlotinib 
or EGFR pathway genes. This is not necessarily incorrect — these genes may 
correlate with drug sensitivity indirectly through tissue type, cell 
proliferation rate, or other confounding factors present in the CCLE dataset. 
Interpreting these top genes would require further biological validation.

### EGFR Pathway Gene Analysis
Erlotinib is a drug that targets a protein called EGFR, which helps control how cancer cells grow and survive. EGFR is part of a larger network of interacting genes and proteins, often referred to as a signaling pathway.

To check whether the model captured anything related to this biological mechanism, a set of well-known genes associated with EGFR signaling was examined in the full ranked gene list:

| Gene | Role | RF Rank | XGB Rank | Percentile (RF) | Percentile (XGB) |
|---|---|---|---|---|---|
| MAPK1 | Downstream effector | 1,732 | 1,719 | 90.8% | 90.9% |
| AKT1 | Downstream effector | 3,559 | 3,137 | 81.2% | 83.4% |
| SRC | Signaling kinase | 6,990 | 7,746 | 63.0% | 59.0% |
| KRAS | Signaling node | 7,932 | 8,692 | 58.0% | 54.0% |
| PIK3CA | Signaling node | 8,194 | 4,560 | 56.6% | 75.9% |
| ERBB4 | EGFR family member | 9,301 | 5,584 | 50.8% | 70.5% |
| EGFR | Drug target | 13,547 | 17,383 | 28.3% | 8.0% |
| MET | Resistance pathway | 14,448 | 17,657 | 23.6% | 6.6% |
| ERBB2 | EGFR family member | 17,290 | 17,758 | 8.5% | 6.0% |
| ERBB3 | EGFR family member | 16,864 | 18,667 | 10.8% | 1.2% |

### Why EGFR Itself Does Not Rank Highly

This is an important and somewhat surprising finding.

### Why EGFR Itself Does Not Rank Highly

This section provides a simple biological interpretation of the model results.

Erlotinib targets the EGFR protein. A natural expectation is that higher EGFR expression would lead to higher sensitivity, and that EGFR would rank as an important predictive gene. However, EGFR ranks in the lower third of all 18,900 genes.

One possible explanation is that gene expression alone does not fully reflect how much a cell depends on EGFR for growth. A cell may express EGFR but still rely on other processes to survive, so blocking EGFR may not have a strong effect.

The dataset used here contains gene expression values, but it does not capture how active these processes are in the cell. As a result, important signals related to drug response may not be fully represented.

This is a known limitation of using expression data alone for predicting drug sensitivity, especially for targeted therapies like Erlotinib (Iorio et al., 2016).

### What the Model Did Learn

Despite EGFR ranking poorly, two related genes (MAPK1 and AKT1) ranked 
consistently in the top 10–20% across both models.

- **MAPK1** (rank ~1,720, top 10%) is involved in cell growth processes 
  connected to EGFR. Higher MAPK1 expression may indicate that these cells 
  are more influenced by EGFR-related activity.
- **AKT1** (rank ~3,100–3,500, top 17–19%) is also involved in cell survival 
  processes linked to EGFR. Higher AKT1 expression may suggest a similar 
  dependence on these signals.

The consistent ranking of these genes across both Random Forest and XGBoost 
is a positive sign. It suggests the models are capturing meaningful patterns 
related to the drug’s mechanism, rather than just random noise, even though 
the overall R² is modest.

### Summary of Findings
The feature importance analysis suggests that:
1. Predictive signal is distributed across many genes, with no single dominant predictor — this is typical for high-dimensional gene expression data
2. EGFR expression itself is not a strong predictor of Erlotinib sensitivity in this dataset, likely because expression alone does not fully reflect how much a cell depends on EGFR
3. Related genes (MAPK1, AKT1) rank higher than EGFR, suggesting the model is capturing broader patterns connected to the drug’s effect

These findings highlight a limitation of using expression data alone for predicting drug sensitivity. Including additional types of biological information would likely improve both predictive performance and interpretability.

### Saved Artifacts
- `top_predictive_genes.csv`
- `egfr_pathway_gene_ranks.csv`

## References

- Iorio, F., et al. (2016). A Landscape of Pharmacogenomic Interactions in 
  Cancer. *Cell*, 166(3), 740–754.
  https://doi.org/10.1016/j.cell.2016.06.017


## Acknowledgements

AI Assistance: Claude (Anthropic) was used for code debugging and literature 
review assistance. All analysis decisions, model selection, and result 
interpretation are the author's own.